In [ ]:
"""
README: YOLO City-Wise Annotation & Street Distribution Exporter

This script scans YOLO-format label files across dataset splits (train/valid/test),
maps each file to its corresponding city and street prefix using filename keywords,
verifies image/label integrity (missing labels, empty labels, unmapped filenames),
prints a detailed summary, and exports the full distribution tables to an Excel (.xlsx) file.

Sheets generated in the Excel workbook:
- City_Summary: Per-city image counts, share (%), per-class annotation counts, and totals.
- Split_Breakdown: Per-city and per-split (train/valid/test) image and class annotation counts.
- Street_Breakdown: Per-street/location prefix image and class annotation counts.
- Validation_Summary: Dataset integrity checks (image/label match, empty labels, unknown files).

How to use:
- Set `base_dir` to your YOLO dataset root directory (containing train/, valid/, test/).
- Set `output_excel_path` to the desired output .xlsx file path.
- Adjust `city_keywords` and `class_names` if your dataset uses different locations or classes.

Requirements:
- Python 3.8+
- openpyxl (`pip install openpyxl`)

Author: Bahadir Akin Akgul
Date: 24.11.2026
"""

import re
from pathlib import Path
from collections import defaultdict, Counter
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# === Settings ===
base_dir = Path("PATH_TO_DATASET_ROOT")
splits = ["train", "valid", "test"]
image_dirname = "images"
label_dirname = "labels"
output_excel_path = Path("citywise_annotation_distribution.xlsx")

# City keyword mapping
city_keywords = {
    "istanbul": [
        "libadiye", "levent", "taksim", "ciragan", "barbaros",
        "dolmabahce", "bagdat", "muallim", "katar",
    ],
    "paris": ["paris-champs"],
    "munich": ["munih"],
    "marseille": ["marsilya"],
}

city_translation = {
    "istanbul": "Istanbul",
    "paris": "Paris",
    "munich": "Munich",
    "marseille": "Marseille",
    "unknown": "Unknown",
}

# Class names (matches data.yaml)
class_names = {0: "pedestrian", 1: "road", 2: "vehicle"}
display_class_order = [2, 0, 1]  # vehicle, pedestrian, road


def get_city(filename: str) -> str:
    name = filename.lower()
    for city, keywords in city_keywords.items():
        if any(k in name for k in keywords):
            return city
    return "unknown"


def get_street_prefix(filename: str) -> str:
    stem = filename.lower()
    m = re.match(r"^([a-z0-9_-]+?)-\d+_jpg", stem)
    if m:
        return m.group(1)
    m2 = re.match(r"^([a-z0-9_-]+)", stem)
    return m2.group(1) if m2 else stem[:20]


# === Result Containers ===
city_image_counts = Counter()
city_class_counts = defaultdict(Counter)
city_split_image_counts = defaultdict(Counter)
city_split_class_counts = defaultdict(lambda: defaultdict(Counter))
street_image_counts = Counter()
street_class_counts = defaultdict(Counter)

split_integrity = {}
empty_label_files = 0
unknown_files = []

# === Counting & Integrity Verification ===
active_splits = [s for s in splits if (base_dir / s / label_dirname).exists()]

for split in active_splits:
    img_dir = base_dir / split / image_dirname
    lbl_dir = base_dir / split / label_dirname

    img_stems = {p.stem for p in img_dir.glob("*") if p.is_file()} if img_dir.exists() else set()
    lbl_files = sorted(lbl_dir.glob("*.txt"))
    lbl_stems = {p.stem for p in lbl_files}

    split_integrity[split] = {
        "images": len(img_stems),
        "labels": len(lbl_stems),
        "imgs_without_lbl": len(img_stems - lbl_stems),
        "lbls_without_img": len(lbl_stems - img_stems),
    }

    for txt_file in lbl_files:
        stem = txt_file.stem
        city = get_city(stem)
        street = get_street_prefix(stem)

        city_image_counts[city] += 1
        city_split_image_counts[city][split] += 1
        street_image_counts[(city, street)] += 1

        if city == "unknown":
            unknown_files.append(f"{split}/{txt_file.name}")

        ann_count = 0
        with open(txt_file, "r", encoding="utf-8") as f:
            for line in f:
                stripped = line.strip()
                if not stripped:
                    continue
                class_id = int(stripped.split()[0])
                if class_id in class_names:
                    city_class_counts[city][class_id] += 1
                    city_split_class_counts[city][split][class_id] += 1
                    street_class_counts[(city, street)][class_id] += 1
                    ann_count += 1

        if ann_count == 0:
            empty_label_files += 1

# === Consistency Assertions ===
total_images = sum(city_image_counts.values())
total_annotations = sum(sum(c.values()) for c in city_class_counts.values())
assert total_images == sum(sum(d.values()) for d in city_split_image_counts.values()), "Split image sum mismatch!"
assert total_images == sum(street_image_counts.values()), "Street image sum mismatch!"
assert total_annotations == sum(sum(c.values()) for c in street_class_counts.values()), "Street annotation sum mismatch!"

# === Console Output ===
print(f"Total Images (Label Files): {total_images:,}")
print(f"Total Annotations:          {total_annotations:,}")
print(f"Empty Label Files:          {empty_label_files}")
print(f"Unmapped (Unknown) Files:   {len(unknown_files)}\n")

ordered_cities = [c for c, _ in city_image_counts.most_common()]
for city in ordered_cities:
    label = city_translation.get(city, city.capitalize())
    img_cnt = city_image_counts[city]
    ann_cnt = sum(city_class_counts[city].values())
    cls_str = ", ".join(
        f"{class_names[cid]}: {city_class_counts[city][cid]:,}"
        for cid in display_class_order
    )
    print(f"{label:10s} | Images: {img_cnt:6,} ({img_cnt / total_images * 100:5.2f}%) | Annotations: {ann_cnt:7,} | {cls_str}")

# === Excel Export ===
wb = Workbook()

header_font = Font(name="Calibri", size=11, bold=True, color="FFFFFF")
header_fill = PatternFill(start_color="1F4E78", end_color="1F4E78", fill_type="solid")
total_font = Font(name="Calibri", size=11, bold=True)
total_fill = PatternFill(start_color="D9E1F2", end_color="D9E1F2", fill_type="solid")
thin_border = Border(
    left=Side(style="thin", color="D9D9D9"),
    right=Side(style="thin", color="D9D9D9"),
    top=Side(style="thin", color="D9D9D9"),
    bottom=Side(style="thin", color="D9D9D9"),
)


def style_sheet(ws, has_total_row=True, percent_cols=(), float_cols=()):
    ws.freeze_panes = "A2"
    for cell in ws[1]:
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = Alignment(horizontal="center", vertical="center")
        cell.border = thin_border

    max_row = ws.max_row
    max_col = ws.max_column
    for r in range(2, max_row + 1):
        is_total = has_total_row and (r == max_row)
        for c in range(1, max_col + 1):
            cell = ws.cell(row=r, column=c)
            cell.border = thin_border
            if is_total:
                cell.font = total_font
                cell.fill = total_fill
            if isinstance(cell.value, (int, float)):
                if c in percent_cols:
                    cell.number_format = "0.00%"
                elif c in float_cols:
                    cell.number_format = "0.00"
                else:
                    cell.number_format = "#,##0"
                cell.alignment = Alignment(horizontal="right", vertical="center")
            else:
                cell.alignment = Alignment(horizontal="left", vertical="center")

    for col_idx in range(1, max_col + 1):
        col_letter = get_column_letter(col_idx)
        max_len = max(
            len(str(ws.cell(row=r, column=col_idx).value or ""))
            for r in range(1, max_row + 1)
        )
        ws.column_dimensions[col_letter].width = max(max_len + 4, 14)


# Sheet 1: City_Summary
ws1 = wb.active
ws1.title = "City_Summary"
ws1.append([
    "City", "Images", "Image Share",
    "vehicle", "pedestrian", "road",
    "Total Annotations", "Annotation Share", "Avg Annotations / Image"
])

for city in ordered_cities:
    label = city_translation.get(city, city.capitalize())
    img_cnt = city_image_counts[city]
    v_cnt = city_class_counts[city][2]
    p_cnt = city_class_counts[city][0]
    r_cnt = city_class_counts[city][1]
    ann_cnt = v_cnt + p_cnt + r_cnt
    ws1.append([
        label,
        img_cnt,
        img_cnt / total_images if total_images else 0,
        v_cnt,
        p_cnt,
        r_cnt,
        ann_cnt,
        ann_cnt / total_annotations if total_annotations else 0,
        ann_cnt / img_cnt if img_cnt else 0,
    ])

tot_v = sum(city_class_counts[c][2] for c in ordered_cities)
tot_p = sum(city_class_counts[c][0] for c in ordered_cities)
tot_r = sum(city_class_counts[c][1] for c in ordered_cities)
ws1.append([
    "Total",
    total_images,
    1.0 if total_images else 0,
    tot_v,
    tot_p,
    tot_r,
    total_annotations,
    1.0 if total_annotations else 0,
    total_annotations / total_images if total_images else 0,
])
style_sheet(ws1, has_total_row=True, percent_cols=(3, 8), float_cols=(9,))

# Sheet 2: Split_Breakdown
ws2 = wb.create_sheet("Split_Breakdown")
ws2.append(["City", "Split", "Images", "vehicle", "pedestrian", "road", "Total Annotations"])

for city in ordered_cities:
    label = city_translation.get(city, city.capitalize())
    for split in active_splits:
        img_cnt = city_split_image_counts[city][split]
        v_cnt = city_split_class_counts[city][split][2]
        p_cnt = city_split_class_counts[city][split][0]
        r_cnt = city_split_class_counts[city][split][1]
        ws2.append([label, split, img_cnt, v_cnt, p_cnt, r_cnt, v_cnt + p_cnt + r_cnt])

ws2.append(["Total", "all", total_images, tot_v, tot_p, tot_r, total_annotations])
style_sheet(ws2, has_total_row=True)

# Sheet 3: Street_Breakdown
ws3 = wb.create_sheet("Street_Breakdown")
ws3.append(["City", "Street Prefix", "Images", "vehicle", "pedestrian", "road", "Total Annotations"])

for (city, street), img_cnt in sorted(
    street_image_counts.items(),
    key=lambda x: (-city_image_counts[x[0][0]], -x[1], x[0][1]),
):
    label = city_translation.get(city, city.capitalize())
    v_cnt = street_class_counts[(city, street)][2]
    p_cnt = street_class_counts[(city, street)][0]
    r_cnt = street_class_counts[(city, street)][1]
    ws3.append([label, street, img_cnt, v_cnt, p_cnt, r_cnt, v_cnt + p_cnt + r_cnt])

ws3.append(["Total", "all", total_images, tot_v, tot_p, tot_r, total_annotations])
style_sheet(ws3, has_total_row=True)

# Sheet 4: Validation_Summary
ws4 = wb.create_sheet("Validation_Summary")
ws4.append(["Check / Metric", "Value"])
ws4.append(["Total Label Files (Images)", total_images])
ws4.append(["Total Annotations", total_annotations])
ws4.append(["Empty Label Files", empty_label_files])
ws4.append(["Unmapped (Unknown) Files", len(unknown_files)])
for split in active_splits:
    info = split_integrity[split]
    ws4.append([f"Split '{split}' - Images", info["images"]])
    ws4.append([f"Split '{split}' - Label Files", info["labels"]])
    ws4.append([f"Split '{split}' - Images without Labels", info["imgs_without_lbl"]])
    ws4.append([f"Split '{split}' - Labels without Images", info["lbls_without_img"]])
style_sheet(ws4, has_total_row=False)

output_excel_path.parent.mkdir(parents=True, exist_ok=True)
wb.save(output_excel_path)
print(f"\nExcel report saved to: {output_excel_path}")
